# Supplementary Code — Brazilian Ethanol Road-Freight Clustering

This notebook is part of the supplementary code package for the manuscript **“Unsupervised Learning for Spatiotemporal Pattern Discovery in Brazilian Ethanol Road Freight Prices: Implications for Sustainable Logistics.”**

**Data are not distributed with this code package.** The private ESALQ-Log freight quotations are third-party data subject to academic-use restrictions. Public contextual data must be downloaded from their original providers. See `../data/README.md` and the package-level `README.md`.

The publication copy contains code and methodological documentation only. Cell outputs were intentionally removed to avoid redistributing restricted data or derivative tables inadvertently.


# Comparativo pós-EDA — 4 escalas temporais × 3 cenários espaciais

Execute primeiro os quatro notebooks atualizados e baixe os arquivos:

- `RESUMO_POS_EDA_MENSAL.csv`
- `RESUMO_POS_EDA_TRIMESTRAL.csv`
- `RESUMO_POS_EDA_SEMESTRAL.csv`
- `RESUMO_POS_EDA_ANUAL.csv`

Depois faça upload desses quatro CSVs neste notebook.

O comparativo usa o **K-Means integrado ampliado k=3** como especificação comum e mantém as métricas do baseline para medir ganho/perda após a inclusão das novas variáveis.


In [ ]:
from pathlib import Path
import os, glob, numpy as np, pandas as pd, matplotlib.pyplot as plt

RESULTS_DIR = Path(os.environ.get("ETHANOL_RESULTS_DIR", ".")).resolve()

def localizar(p):
    candidates = glob.glob(str(RESULTS_DIR / p)) + glob.glob(p)
    if not candidates:
        raise FileNotFoundError(p)
    return candidates[0]

arquivos = [
    localizar("RESUMO_POS_EDA_MENSAL.csv"),
    localizar("RESUMO_POS_EDA_TRIMESTRAL.csv"),
    localizar("RESUMO_POS_EDA_SEMESTRAL.csv"),
    localizar("RESUMO_POS_EDA_ANUAL.csv")
]

comparacao = pd.concat([pd.read_csv(a) for a in arquivos], ignore_index=True)
display(comparacao.round(4))


In [ ]:
# Ganho/perda versus baseline

if "Silhouette_k3" in comparacao.columns:
    comparacao["Delta_Silhouette_vs_baseline"] = (
        comparacao["Silhouette"] - comparacao["Silhouette_k3"]
    )

if "Davies_k3" in comparacao.columns:
    comparacao["Delta_Davies_vs_baseline"] = (
        comparacao["Davies_Bouldin"] - comparacao["Davies_k3"]
    )

display(
    comparacao[
        [c for c in [
            "Agregacao","Cenario",
            "Silhouette","Silhouette_k3","Delta_Silhouette_vs_baseline",
            "Davies_Bouldin","Davies_k3","Delta_Davies_vs_baseline",
            "Calinski_Harabasz","Menor_cluster_pct",
            "Coerencia_vizinhos_pct","Moran_I_frete","Moran_p_frete",
            "ARI_vs_baseline_k3"
        ] if c in comparacao.columns]
    ].round(4)
)


In [ ]:
# Elegibilidade: evita clusters residuais.
elegiveis = comparacao[comparacao["Menor_cluster_pct"] >= 5].copy()

# Ranking multicritério transparente.
# Não é um teste inferencial; serve como apoio à seleção do modelo.
def minmax(s, maior_melhor=True):
    s = pd.to_numeric(s, errors="coerce")
    if s.max() == s.min():
        z = pd.Series(1.0, index=s.index)
    else:
        z = (s - s.min()) / (s.max() - s.min())
    return z if maior_melhor else 1-z

elegiveis["S_sil"] = minmax(elegiveis["Silhouette"], True)
elegiveis["S_db"] = minmax(elegiveis["Davies_Bouldin"], False)
elegiveis["S_ch"] = minmax(elegiveis["Calinski_Harabasz"], True)
elegiveis["S_bal"] = minmax(elegiveis["Menor_cluster_pct"], True)
elegiveis["S_sp"] = minmax(elegiveis["Coerencia_vizinhos_pct"], True)

if "Pct_observado" in elegiveis.columns:
    elegiveis["S_obs"] = minmax(elegiveis["Pct_observado"], True)
else:
    elegiveis["S_obs"] = 1.0

# Pesos mantêm Silhouette e Davies como eixos centrais,
# mas consideram equilíbrio, observação e coerência espacial.
elegiveis["Score_multicriterio"] = (
    0.30*elegiveis["S_sil"]
    + 0.20*elegiveis["S_db"]
    + 0.15*elegiveis["S_ch"]
    + 0.15*elegiveis["S_bal"]
    + 0.10*elegiveis["S_obs"]
    + 0.10*elegiveis["S_sp"]
)

ranking = elegiveis.sort_values(
    ["Score_multicriterio","Silhouette","Davies_Bouldin"],
    ascending=[False,False,True]
).reset_index(drop=True)

ranking.insert(0, "Posicao", np.arange(1,len(ranking)+1))
display(ranking.round(4))

lider = ranking.iloc[0]
print("\nMODELO PRINCIPAL PÓS-EDA:")
print(lider[["Agregacao","Cenario","Silhouette","Davies_Bouldin","Menor_cluster_pct","Coerencia_vizinhos_pct","Score_multicriterio"]])


In [ ]:
# Robustez temporal e espacial

agreg_principal = lider["Agregacao"]
cenario_principal = lider["Cenario"]

rob_temporal = ranking[
    (ranking["Cenario"] == cenario_principal)
    & (ranking["Agregacao"] != agreg_principal)
].head(1)

rob_espacial = ranking[
    (ranking["Agregacao"] == agreg_principal)
    & (ranking["Cenario"] != cenario_principal)
].head(1)

print("Robustez temporal:")
display(rob_temporal.round(4))

print("Robustez espacial:")
display(rob_espacial.round(4))


In [ ]:
# Gráficos comparativos dos 12 modelos

fig, axes = plt.subplots(2,2, figsize=(14,10))

for cenario, g in comparacao.groupby("Cenario"):
    axes[0,0].plot(g["Agregacao"], g["Silhouette"], marker="o", label=cenario)
axes[0,0].set_title("Silhouette — K-Means integrado ampliado k=3")
axes[0,0].legend()
axes[0,0].grid(alpha=.3)

for cenario, g in comparacao.groupby("Cenario"):
    axes[0,1].plot(g["Agregacao"], g["Davies_Bouldin"], marker="o", label=cenario)
axes[0,1].set_title("Davies-Bouldin — menor é melhor")
axes[0,1].legend()
axes[0,1].grid(alpha=.3)

for cenario, g in comparacao.groupby("Cenario"):
    axes[1,0].plot(g["Agregacao"], g["Coerencia_vizinhos_pct"], marker="o", label=cenario)
axes[1,0].set_title("Coerência espacial dos clusters")
axes[1,0].set_ylabel("% vizinhos no mesmo cluster")
axes[1,0].legend()
axes[1,0].grid(alpha=.3)

for cenario, g in comparacao.groupby("Cenario"):
    axes[1,1].plot(g["Agregacao"], g["Menor_cluster_pct"], marker="o", label=cenario)
axes[1,1].axhline(5, linestyle="--", linewidth=1)
axes[1,1].set_title("Tamanho do menor cluster")
axes[1,1].set_ylabel("% das unidades")
axes[1,1].legend()
axes[1,1].grid(alpha=.3)

plt.tight_layout()
plt.savefig("COMPARATIVO_POS_EDA_12_MODELOS.png", dpi=300, bbox_inches="tight")
plt.show()

ranking.to_csv("RANKING_POS_EDA_12_MODELOS.csv", index=False, encoding="utf-8-sig")
comparacao.to_csv("COMPARACAO_POS_EDA_12_MODELOS.csv", index=False, encoding="utf-8-sig")


## Regra de decisão

O ranking multicritério é uma ferramenta de decisão, não um teste inferencial. A conclusão final deve considerar:

1. ausência de cluster residual (<5%);
2. Silhouette e Davies-Bouldin;
3. Calinski-Harabasz;
4. cobertura observacional;
5. coerência espacial;
6. preservação de granularidade logística;
7. estabilidade em DTW, Moran/LISA e cenários de robustez.

Os modelos somente com frete permanecem como baseline para medir se as novas variáveis realmente acrescentam estrutura aos clusters.


# 7. Comparativo final após as final robustness checks

**Esta seção substitui o ranking anterior como base para a decisão metodológica.**

Antes de executar este notebook, execute os quatro notebooks ajustados e faça upload dos arquivos:

- `SUMMARY_FINAL_MENSAL.csv`
- `SUMMARY_FINAL_TRIMESTRAL.csv`
- `SUMMARY_FINAL_SEMESTRAL.csv`
- `SUMMARY_FINAL_ANUAL.csv`
- `ROBUSTNESS_DENSIDADE_*.csv`
- `ROBUSTNESS_DTW_DBA_*.csv`
- `ROBUSTNESS_KNN_SENSIBILIDADE_*.csv`
- `ROBUSTNESS_KNN_ASSOCIACAO_*.csv`
- `ROBUSTNESS_LAG_CANA_*.csv`

O objetivo agora é responder explicitamente às questões levantadas na revisão:

1. Qual escala temporal é melhor **em média**?
2. Qual cenário espacial é melhor **em média**?
3. Quais são as piores escolhas?
4. Ter mais dados observados implica melhores clusters?
5. Maior dependência de KNN está associada a métricas piores?
6. K-Means continua competitivo frente a DBSCAN, HDBSCAN e DTW K-Means/DBA?


In [ ]:
# CROSS-SCENARIO ROBUSTNESS 1 — carregar os novos resultados

import os, glob, numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import spearmanr

def localizar_um(padrao):
    c = sorted(set(glob.glob(str(RESULTS_DIR / padrao)) + glob.glob(padrao)))
    if not c:
        raise FileNotFoundError(padrao)
    return c[0]

ordem_agreg = ["Mensal","Trimestral","Semestral","Anual"]

arquivos_resumo = [localizar_um(f"SUMMARY_FINAL_{a.upper()}.csv") for a in ordem_agreg]
resumo_prof = pd.concat([pd.read_csv(a) for a in arquivos_resumo],ignore_index=True)

arquivos_den = [localizar_um(f"ROBUSTNESS_DENSIDADE_{a.upper()}.csv") for a in ordem_agreg]
densidade_prof = pd.concat([pd.read_csv(a) for a in arquivos_den],ignore_index=True)

arquivos_dtw = [localizar_um(f"ROBUSTNESS_DTW_DBA_{a.upper()}.csv") for a in ordem_agreg]
dtw_prof = pd.concat([pd.read_csv(a) for a in arquivos_dtw],ignore_index=True)

arquivos_knn = [localizar_um(f"ROBUSTNESS_KNN_SENSIBILIDADE_{a.upper()}.csv") for a in ordem_agreg]
knn_prof = pd.concat([pd.read_csv(a) for a in arquivos_knn],ignore_index=True)

arquivos_knn_assoc = [localizar_um(f"ROBUSTNESS_KNN_ASSOCIACAO_{a.upper()}.csv") for a in ordem_agreg]
knn_assoc_prof = pd.concat([pd.read_csv(a) for a in arquivos_knn_assoc],ignore_index=True)

arquivos_lag = [localizar_um(f"ROBUSTNESS_LAG_CANA_{a.upper()}.csv") for a in ordem_agreg]
lag_prof = pd.concat([pd.read_csv(a) for a in arquivos_lag],ignore_index=True)

resumo_prof["Agregacao"] = pd.Categorical(resumo_prof["Agregacao"],categories=ordem_agreg,ordered=True)
display(resumo_prof.round(4))


In [ ]:
# CROSS-SCENARIO ROBUSTNESS 2 — desempenho MÉDIO por escala temporal e por cenário espacial

metricas_media = [
    "Silhouette_k3","Davies_k3","Calinski_k3",
    "Pct_observado_final","Pct_KNN_final","Unidades_clusterizadas",
    "Silhouette_nucleo","Davies_nucleo",
    "Silhouette_ampliado","Davies_ampliado"
]
metricas_media = [c for c in metricas_media if c in resumo_prof.columns]

media_temporal = resumo_prof.groupby("Agregacao",observed=True)[metricas_media].mean().reset_index()
media_espacial = resumo_prof.groupby("Cenario")[metricas_media].mean().reset_index()

print("MÉDIA POR ESCALA TEMPORAL")
display(media_temporal.round(4))

print("MÉDIA POR CENÁRIO ESPACIAL")
display(media_espacial.round(4))

fig,axes=plt.subplots(1,2,figsize=(13,4.5))
axes[0].plot(media_temporal["Agregacao"].astype(str),media_temporal["Silhouette_k3"],marker="o")
axes[0].set_title("Silhouette baseline médio por escala temporal")
axes[0].set_ylabel("Silhouette médio")
axes[0].grid(alpha=.3)

axes[1].bar(media_espacial["Cenario"],media_espacial["Silhouette_k3"])
axes[1].set_title("Silhouette baseline médio por cenário espacial")
axes[1].set_ylabel("Silhouette médio")
plt.tight_layout()
plt.savefig("ROBUSTNESS_Medias_Escala_Cenario.png",dpi=300,bbox_inches="tight")
plt.show()


In [ ]:
# CROSS-SCENARIO ROBUSTNESS 3 — melhores e piores especificações

rank_base = resumo_prof.sort_values(
    ["Silhouette_k3","Davies_k3"],ascending=[False,True]
).copy()

print("3 MELHORES BASELINES k=3")
display(rank_base.head(3)[[
    "Agregacao","Cenario","Silhouette_k3","Davies_k3","Calinski_k3",
    "Pct_observado_final","Pct_KNN_final","Unidades_clusterizadas"
]].round(4))

print("3 PIORES BASELINES k=3")
display(rank_base.tail(3)[[
    "Agregacao","Cenario","Silhouette_k3","Davies_k3","Calinski_k3",
    "Pct_observado_final","Pct_KNN_final","Unidades_clusterizadas"
]].round(4))

# Modelos integrados — piores/melhores pelo núcleo e ampliado.
for modelo in ["nucleo","ampliado"]:
    sil=f"Silhouette_{modelo}"
    db=f"Davies_{modelo}"
    if sil in resumo_prof:
        rr=resumo_prof.sort_values([sil,db],ascending=[False,True])
        print(f"\nMELHORES — {modelo}")
        display(rr.head(3)[["Agregacao","Cenario",sil,db]].round(4))
        print(f"PIORES — {modelo}")
        display(rr.tail(3)[["Agregacao","Cenario",sil,db]].round(4))


In [ ]:
# CROSS-SCENARIO ROBUSTNESS 4 — "ter mais dados = melhores modelos?" e influência do KNN

analises = []
for x in ["Pct_observado_final","Pct_KNN_final","Unidades_clusterizadas"]:
    for y in ["Silhouette_k3","Davies_k3"]:
        rho,p = spearmanr(resumo_prof[x],resumo_prof[y],nan_policy="omit")
        analises.append({"X":x,"Y":y,"Spearman_rho":rho,"p_value":p})

dados_vs_metricas = pd.DataFrame(analises)
display(dados_vs_metricas.round(4))

fig,axes=plt.subplots(1,2,figsize=(12,4.5))
axes[0].scatter(resumo_prof["Pct_observado_final"],resumo_prof["Silhouette_k3"])
for _,r in resumo_prof.iterrows():
    axes[0].annotate(f"{r['Agregacao']}-{r['Cenario']}",
                     (r["Pct_observado_final"],r["Silhouette_k3"]),fontsize=7)
axes[0].set_xlabel("% originalmente observado")
axes[0].set_ylabel("Silhouette baseline k=3")
axes[0].set_title("Mais dados observados = melhor cluster?")
axes[0].grid(alpha=.3)

axes[1].scatter(resumo_prof["Pct_KNN_final"],resumo_prof["Silhouette_k3"])
for _,r in resumo_prof.iterrows():
    axes[1].annotate(f"{r['Agregacao']}-{r['Cenario']}",
                     (r["Pct_KNN_final"],r["Silhouette_k3"]),fontsize=7)
axes[1].set_xlabel("% KNN na matriz final")
axes[1].set_ylabel("Silhouette baseline k=3")
axes[1].set_title("Dependência de KNN × qualidade do clustering")
axes[1].grid(alpha=.3)

plt.tight_layout()
plt.savefig("ROBUSTNESS_Dados_Imputacao_vs_Qualidade.png",dpi=300,bbox_inches="tight")
plt.show()

print("SENSIBILIDADE DIRETA A KNN")
display(knn_prof.sort_values(["Agregacao","Cenario","Tratamento"]).round(4))
print("ASSOCIAÇÃO PCT KNN × SILHUETA INDIVIDUAL")
display(knn_assoc_prof.round(4))


In [ ]:
# CROSS-SCENARIO ROBUSTNESS 5 — K-Means vs DBSCAN vs HDBSCAN vs DTW K-Means/DBA

# K-Means baseline k=3
km = resumo_prof[["Agregacao","Cenario","Silhouette_k3","Davies_k3"]].copy()
km = km.rename(columns={"Silhouette_k3":"Silhouette","Davies_k3":"Davies_Bouldin"})
km["Algoritmo"]="KMeans_Euclidiano"
km["Representacao"]="Frete_only"
km["Ruido_pct"]=0.0

# Melhores soluções de densidade na representação Frete_only
den = densidade_prof[densidade_prof["Representacao"]=="Frete_only"].copy()
den = den[["Agregacao","Cenario","Algoritmo","Representacao","Silhouette","Davies_Bouldin","Ruido_pct","N_clusters"]]

# DTW/DBA frete k=3
dt = dtw_prof[dtw_prof["Modelo"]=="DTW_DBA_frete"].copy()
dt = dt.rename(columns={"Silhouette_DTW":"Silhouette"})
dt["Algoritmo"]="DTW_KMeans_DBA"
dt["Representacao"]="Frete_only"
dt["Davies_Bouldin"]=np.nan
dt["Ruido_pct"]=0.0
dt = dt[["Agregacao","Cenario","Algoritmo","Representacao","Silhouette","Davies_Bouldin","Ruido_pct"]]

alg_comp = pd.concat([km,den,dt],ignore_index=True,sort=False)
display(alg_comp.sort_values(["Agregacao","Cenario","Silhouette"],ascending=[True,True,False]).round(4))

media_alg = alg_comp.groupby("Algoritmo").agg(
    Silhouette_medio=("Silhouette","mean"),
    Silhouette_mediano=("Silhouette","median"),
    Ruido_medio_pct=("Ruido_pct","mean"),
    N_combinacoes=("Silhouette","count")
).reset_index().sort_values("Silhouette_medio",ascending=False)
print("DESEMPENHO MÉDIO DOS ALGORITMOS")
display(media_alg.round(4))

fig,ax=plt.subplots(figsize=(10,5))
ax.bar(media_alg["Algoritmo"],media_alg["Silhouette_medio"])
ax.set_ylabel("Silhouette médio")
ax.set_title("Comparação média dos algoritmos de clustering")
ax.tick_params(axis="x",rotation=20)
plt.tight_layout()
plt.savefig("ROBUSTNESS_Comparacao_Media_Algoritmos.png",dpi=300,bbox_inches="tight")
plt.show()

print("""
NOTA DE INTERPRETAÇÃO:
- K-Means e DBSCAN/HDBSCAN usam a representação vetorial padronizada.
- Em DBSCAN/HDBSCAN, Silhouette é calculada apenas para pontos não classificados como ruído;
  o percentual de ruído deve ser analisado junto.
- DTW K-Means usa uma geometria temporal diferente; sua Silhouette é baseada na distância DTW.
- Portanto, o 'melhor modelo' não deve ser escolhido apenas pelo maior número de Silhouette:
  cobertura, ruído, estabilidade, interpretabilidade e aderência à pergunta também importam.
""")


In [ ]:
# CROSS-SCENARIO ROBUSTNESS 6 — sensibilidade ao lag da produção de cana

display(lag_prof.round(4))

lag_media = (
    lag_prof.groupby(["Agregacao","Lag"])["Spearman_rho"]
    .mean().reset_index()
)
display(lag_media.round(4))

fig,ax=plt.subplots(figsize=(11,4.5))
for lag,g in lag_media.groupby("Lag"):
    ax.plot(g["Agregacao"],g["Spearman_rho"],marker="o",label=lag)
ax.axhline(0,linewidth=1)
ax.set_title("Associação frete × cana: contemporânea versus lag de 1 ano")
ax.set_ylabel("Spearman rho médio entre cenários")
ax.legend()
ax.grid(alpha=.3)
plt.tight_layout()
plt.savefig("ROBUSTNESS_Lag_Cana_Comparativo.png",dpi=300,bbox_inches="tight")
plt.show()


## 14. Matriz de uso/decisão por escala temporal × agregação espacial

A tabela abaixo não é um resultado estatístico; é um **guia de interpretação gerencial** para conectar cada combinação ao tipo de decisão que ela pode apoiar.

| Escala temporal ↓ / espacial → | C1 - Município→município | C2 - Região Imediata | C3 - Região Intermediária |
|---|---|---|---|
| **Mensal** | Monitoramento operacional e heterogeneidade fina de rotas | Ajustes operacionais regionais | Sinais mensais macroregionais, com maior suavização |
| **Trimestral** | Benchmarking tático de corredores e sazonalidade - boa preservação logística | Coordenação tática regional | Comparação trimestral macroregional |
| **Semestral** | Avaliação tático-estratégica e robustez espacial | Planejamento regional semestral | Tendências estruturais com forte suavização |
| **Anual** | Benchmarking estratégico de longo prazo, com perda intra-anual | Planejamento estratégico regional | Visão macroestratégica; maior risco de ocultar heterogeneidade |

A escolha final deve combinar métricas, suporte observacional e a decisão que se deseja apoiar.


In [ ]:
# CROSS-SCENARIO ROBUSTNESS 7 — exportações finais para os relatórios

media_temporal.to_csv("ROBUSTNESS_MEDIA_POR_ESCALA_TEMPORAL.csv",index=False,encoding="utf-8-sig")
media_espacial.to_csv("ROBUSTNESS_MEDIA_POR_CENARIO_ESPACIAL.csv",index=False,encoding="utf-8-sig")
dados_vs_metricas.to_csv("ROBUSTNESS_DADOS_IMPUTACAO_VS_METRICAS.csv",index=False,encoding="utf-8-sig")
alg_comp.to_csv("ROBUSTNESS_COMPARACAO_ALGORITMOS.csv",index=False,encoding="utf-8-sig")
media_alg.to_csv("ROBUSTNESS_MEDIA_ALGORITMOS.csv",index=False,encoding="utf-8-sig")
lag_media.to_csv("ROBUSTNESS_LAG_CANA_MEDIA.csv",index=False,encoding="utf-8-sig")

print("Exportações concluídas. Após executar este notebook, baixe também estes CSVs para o relatório final.")


## 15. Regra revisada de decisão

Após a final robustness review, a decisão final não deve ser baseada apenas no **melhor valor absoluto**.

A interpretação deve responder conjuntamente:

- qual agregação temporal é melhor **em média** e qual é pior;
- qual cenário espacial é melhor **em média** e qual é pior;
- se maior quantidade de dados observados está associada a melhor clusterização;
- se maior dependência de KNN altera os resultados;
- se K-Means permanece competitivo frente a DBSCAN/HDBSCAN;
- se DTW/DBA acrescenta informação temporal relevante;
- se a estrutura espacial observada em Moran/LISA é coerente com a logística descrita na literatura.

**Somente depois desses outputs executados será reescrito o Relatório Executivo e o Relatório 4.**
